# Week 06 — BBO capstone driver

Round 6. Every function gets a ±0.003 perturbation of its W5 point.

**Stated intent:** confirm W5's readings and extract a local gradient from each pair.

**What it actually does:** returns all eight values within 2% of W5. At a step of 0.003 on a surface with length-scale of order 0.1, the finite difference is dominated by whatever numerical noise the portal carries. This round buys close to nothing, and I repeat the mistake twice more before diagnosing it.

Recorded here without softening, because the budget is measured in **rounds** and this is one of four donated back.

In [ ]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
# Walk up until bbo.py is found, so the notebook runs from anywhere in the repo.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "bbo.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
import numpy as np
import pandas as pd
import bbo

WEEK = 6
PRIOR = WEEK - 1          # data state this round was proposed from
SEED = 6
OUTDIR = f"outputs/week{WEEK:02d}"; os.makedirs(OUTDIR, exist_ok=True)

# What each function is getting this round, and why.
PLAN = {
    1: '±0.003 confirmation of W5',
    2: '±0.003 confirmation of W5',
    3: '±0.003 confirmation of W5',
    4: '±0.003 confirmation of W5',
    5: '±0.003 confirmation of W5',
    6: '±0.003 confirmation of W5',
    7: '±0.003 confirmation of W5',
    8: '±0.003 confirmation of W5',
}
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], move=PLAN[f]) for f in bbo.FUNC_IDS])


## 1. Data — the state this round was proposed from

Best point on record per function, truncated to rounds ≤ 5. Nothing below this cell may look at later rounds.


In [ ]:
# The ledger comes FIRST every round: the best point on record, not the latest one.
led = bbo.ledger(up_to=PRIOR)
led["best"] = led["best"].map(lambda v: f"{v:.6g}")
led["x"] = led["x"].map(bbo.submission)
led


## 2. Proposals — ±0.003 around W5

The step column is the diagnostic. Compare it against the length-scales the GP fits below: if the step is orders of magnitude smaller, expect no signal.

In [ ]:
proposals = {
    1: np.array([0.051203, 0.134018]),
    2: np.array([0.046215, 0.140382]),
    3: np.array([0.492341, 0.156982, 0.898517]),
    4: np.array([0.566438, 0.446519, 0.399884, 0.285792]),
    5: np.array([0.278106, 0.701274, 0.061098, 0.632517]),
    6: np.array([0.451687, 0.913498, 0.334419, 0.208271, 0.610063]),
    7: np.array([0.656251, 0.845498, 0.279274, 0.498842, 0.78991, 0.391421]),
    8: np.array([0.018683, 0.255785, 0.161747, 0.328043, 0.783074, 0.214943, 0.955801, 0.074683]),
}

prev = {fid: np.array(bbo.HISTORY[5][fid][0]) for fid in bbo.FUNC_IDS}
rows = []
for fid in bbo.FUNC_IDS:
    step = float(np.linalg.norm(proposals[fid] - prev[fid]))
    ls = np.atleast_1d(bbo.fit(fid, up_to=PRIOR).gp.kernel_.k1.k2.length_scale)
    rows.append(dict(func=f"F{fid}", step=round(step, 5),
                     median_lengthscale=round(float(np.median(ls)), 4),
                     ratio=round(step / float(np.median(ls)), 5)))
pd.DataFrame(rows)


### Surrogate trust check

Run before reading any acquisition value, not after.


In [ ]:
# Is each surrogate worth listening to? LOO R2 < 0 means it is worse than
# predicting the mean, and any acquisition value built on it is arbitrary.
rows = []
for fid in bbo.FUNC_IDS:
    X, y, _ = bbo.load(fid, up_to=PRIOR)
    r2 = bbo.fit(fid, up_to=PRIOR).loo_r2() if len(y) >= 4 else float("nan")
    rows.append(dict(func=f"F{fid}", n_data=len(y), loo_r2=round(r2, 3),
                     verdict="broken" if r2 < 0 else "usable" if r2 == r2 else "too few points"))
pd.DataFrame(rows)


### Anchor audit


In [ ]:
ANCHOR = {
    1: [0.048421, 0.137247],
    2: [0.048699, 0.137537],
    3: [0.489658, 0.159666, 0.895963],
    4: [0.569121, 0.44372, 0.402671, 0.283008],
    5: [0.275418, 0.703962, 0.058314, 0.62974],
    6: [0.448901, 0.916285, 0.331657, 0.205493, 0.612847],
    7: [0.659037, 0.842715, 0.27649, 0.501628, 0.787126, 0.394208],
    8: [0.015899, 0.258569, 0.158963, 0.325259, 0.785858, 0.212159, 0.958585, 0.071899],
}
# Anchor audit: is each proposal being generated from the best point on record?
# This is the check whose absence cost the campaign most of its final score.
for fid in bbo.FUNC_IDS:
    w = bbo.anchor_check(fid, np.array(ANCHOR[fid], float), up_to=PRIOR)
    print(f"F{fid}: {w if w else 'anchored on best-known point'}")


## 3. Visualise

Best-so-far trajectory per function, truncated to the data available this round.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), bbo.FUNC_IDS):
    try:
        _, y, rounds = bbo.load(fid, up_to=PRIOR)
    except ValueError:
        ax.set_title(f"F{fid}: no data"); continue
    ax.plot(rounds, y, "o", ms=4, alpha=.55)
    ax.plot(rounds, np.maximum.accumulate(y), "-", lw=2)
    ax.set_title(f"F{fid} (d={bbo.DIMS[fid]})", fontsize=9)
    ax.tick_params(labelsize=7); ax.set_xlabel("round", fontsize=8)
fig.suptitle(f"Best so far through round {PRIOR}", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/trajectories.png", dpi=140)
plt.show()


## 4. Submission strings


In [ ]:
# Portal format: six decimals, dash-separated, one line per function, no labels.
for fid in bbo.FUNC_IDS:
    print(bbo.submission(proposals[fid]))


## 5. After the portal returns each y

Returns recorded below and folded into `bbo.HISTORY` so the next round sees them.


In [ ]:
# Week 6 portal returns - already folded into bbo.HISTORY.
# returned_y = {
#     1: -6.377968075830329e-154,
#     2: 0.054542698829435535,
#     3: -0.09678129158071659,
#     4: -4.15418124708534,
#     5: 2.435323698188698,
#     6: -2.0214828656562784,
#     7: 0.16998035132007513,
#     8: 8.6893948626831,
# }
#
# Zero new bests. Every return within 2% of W5. The ratio column above is the whole
# story: the step is two to three orders below the fitted length-scale, so the pair
# is a repeated measurement, not a finite difference.
#
# for fid, y in returned_y.items():
#     bbo.append_result(fid, proposals[fid], y, rnd=WEEK)
